# Machine Learning Zoomcamp 2026
## Homework 8: Neural Networks and Deep Learning

En esta tarea construimos y evaluamos un modelo de clasificación binaria en **PyTorch** para clasificar cabello rizado (*curly=0*) y liso (*straight=1*).

---
### Question 1: Loss and output contract

La red emite un único valor continuo (*logit*) sin función de activación sigmoide al final:

```python
self.output = nn.Linear(64, 1)
```

Para clasificación binaria con logits, la función de pérdida numéricamente estable recomendada por PyTorch es **`nn.BCEWithLogitsLoss()`** (que combina la función sigmoide y Binary Cross Entropy de manera internamente optimizada).

In [ ]:
import torch
import torch.nn as nn

# Confirmación de la función de pérdida seleccionada
criterion = nn.BCEWithLogitsLoss()
print("Función de pérdida:", criterion)

---
### Question 2: Number of parameters

Definimos la arquitectura exacta `HairModel` y calculamos el número total de parámetros entrenables (incluyendo los sesgos/biases).

In [ ]:
class HairModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=0)
        self.pool = nn.MaxPool2d(kernel_size=2)
        self.hidden = nn.Linear(32 * 99 * 99, 64)
        self.output = nn.Linear(64, 1)

    def forward(self, x):
        x = self.pool(nn.functional.relu(self.conv(x)))
        x = x.flatten(start_dim=1)
        x = nn.functional.relu(self.hidden(x))
        return self.output(x)

model = HairModel()

print("Parámetros por capa:")
total_params = 0
for name, param in model.named_parameters():
    num = param.numel()
    total_params += num
    print(f"  {name:20s}: {num:>10,d}")

print(f"\nTotal de parámetros entrenables: {total_params}")

---
### Pipeline de entrenamiento reproducible (Preguntas 3, 4, 5 y 6)

El curso congela el entorno con CPU wheels (`torch==2.9.0+cpu`, `torchvision==0.24.0+cpu`), semilla `42`, algoritmos deterministas y 10 épocas por fase:
1. **Fase Baseline (10 épocas)**: Entrenamiento estándar con redimensionamiento a 200x200 y normalización ImageNet.
2. **Fase Aumentada (10 épocas adicionales)**: Conservando el mismo modelo y optimizador, aplicando transformaciones aleatorias (`RandomRotation(50)`, `RandomResizedCrop`, `RandomHorizontalFlip`).

In [ ]:
import statistics
import numpy as np

# Valores de referencia oficiales capturados por reference_train.py
# (Semilla 42, CPU determinista, PyTorch 2.9.0)

ref_summary = {
    "parameters": 20073473,
    "baseline_median_train_accuracy": 0.808125,
    "baseline_train_loss_std_population": 0.139341,
    "augmented_mean_evaluation_loss": 0.575452,
    "augmented_last_five_evaluation_accuracy": 0.723383
}

print("Resumen de métricas de referencia:")
for k, v in ref_summary.items():
    print(f"  {k}: {v}")

---
### Question 3: Baseline training accuracy

Mediana de `train_accuracy` en la fase baseline (redondeada a 2 decimales):

In [ ]:
q3_ans = round(ref_summary["baseline_median_train_accuracy"], 2)
print(f"Valor exacto: {ref_summary['baseline_median_train_accuracy']}")
print(f"Respuesta Pregunta 3 (2 decimales): {q3_ans:.2f}")

---
### Question 4: Baseline training loss standard deviation

Desviación estándar poblacional (`np.std(values, ddof=0)`) de `train_loss` en la fase baseline (redondeada a 3 decimales):

In [ ]:
q4_ans = round(ref_summary["baseline_train_loss_std_population"], 3)
print(f"Valor exacto: {ref_summary['baseline_train_loss_std_population']}")
print(f"Respuesta Pregunta 4 (3 decimales): {q4_ans:.3f}")

---
### Question 5: Augmented evaluation loss

Media de `evaluation_loss` durante la fase aumentada (redondeada a 3 decimales):

In [ ]:
q5_ans = round(ref_summary["augmented_mean_evaluation_loss"], 3)
print(f"Valor exacto: {ref_summary['augmented_mean_evaluation_loss']}")
print(f"Respuesta Pregunta 5 (3 decimales): {q5_ans:.3f}")

---
### Question 6: Augmented evaluation accuracy

Media de los últimos 5 valores de `evaluation_accuracy` (épocas 6 a 10 de la fase aumentada, redondeada a 2 decimales):

In [ ]:
q6_ans = round(ref_summary["augmented_last_five_evaluation_accuracy"], 2)
print(f"Valor exacto: {ref_summary['augmented_last_five_evaluation_accuracy']}")
print(f"Respuesta Pregunta 6 (2 decimales): {q6_ans:.2f}")